# İnce ayar — Colab / Kaggle defteri

Bu defter `havuz/ince-ayar` temelini çeker, kurar ve **hazır ayarla** (`colab-t4`) komutları sırayla çalıştırır.
Bütün iş komut satırındadır; defter yalnız onları çağırır. Ayrıntı: `README.md` ve `AGENTS.md`.

1. Üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. Hücreleri yukarıdan aşağı çalıştırın. Kendi verinizi kullanmak için dosyayı yükleyip komutlara `-D veri.egitim=DOSYA` ekleyin.
3. **Lisans:** taban modelin ve verinizin lisansını `katalog/modeller.yaml` içinde denetleyin.

> Not: Bu defterdeki komutların biçimi testle denetlenir (`tests/test_notebook.py`); T4 üzerinde **uçtan uca çalıştırılması doğrulanmadı**
> (geliştirme makinesinde GPU yoktu). İlk çalıştırmada sorun çıkarsa hata metnini olduğu gibi bir modele yapıştırın.

In [ ]:
# GPU var mı? (Tesla T4 görmelisiniz)
!nvidia-smi

In [ ]:
# Depoyu çek ve temeli kur. Depo gizliyse adresi erişim belirteciyle yazın ya da depoyu zip olarak yükleyin.
DEPO = "https://github.com/yilmazfatih-cyber/Deneme"
TEMEL = "havuz/ince-ayar"
!git clone --depth 1 {DEPO} depo
%cd depo/{TEMEL}
!pip install -q -e ".[egitim,gomme,onnx]"

## 1. Sohbet modeli (LoRA)

Önce veriyi doğrulayın: hatalar satır numarasıyla gelir, ilk örneğin **kayıp maskesi** ⟦…⟧ içinde gösterilir.
Sonra kuru çalıştırma planı ve tahmini belleği yazar; eğitim başlamaz.

In [ ]:
!python -m ince_ayar sohbet dogrula --hazir colab-t4
!python -m ince_ayar sohbet egit --hazir colab-t4 --kuru

In [ ]:
# Eğitim (örnek veri çok küçüktür; gerçek iş için -D veri.egitim=veri/sohbet.jsonl verin)
!python -m ince_ayar sohbet egit --hazir colab-t4

In [ ]:
# Değerlendirme: ince ayarlı model ile taban model aynı doğrulama verisinde
!python -m ince_ayar sohbet degerlendir --hazir colab-t4 --karsilastir
!python -m ince_ayar sohbet tahmin --model ciktilar/sohbet-t4 --mesaj "Kargom ne zaman gelir?"

## 2. Birleştir → GGUF → Ollama

Bağdaştırıcı taban modele katılır; llama.cpp dönüştürücüsü GGUF üretir; `Modelfile` yazılır.
llama.cpp'nin gereksinim dosyası transformers'ı eski sürüme sabitlediği için onu **kurmayın**; yalnız `sentencepiece` yeter.

In [ ]:
!python -m ince_ayar sohbet birlestir --model ciktilar/sohbet-t4 --cikti ciktilar/sohbet-t4-birlesik
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q sentencepiece
!python -m ince_ayar sohbet disa-aktar --model ciktilar/sohbet-t4-birlesik --cikti ciktilar/gguf --llama-cpp llama.cpp --ad benim-modelim

## 3. Metin sınıflandırma → ONNX

Eğitim, doğrulamada eşik ayarı, test raporu (doğruluk, makro F1, karışıklık matrisi), sonra ONNX (+ int8).

In [ ]:
!python -m ince_ayar siniflandirma dogrula --hazir colab-t4
!python -m ince_ayar siniflandirma egit --hazir colab-t4
!python -m ince_ayar siniflandirma disa-aktar --hazir colab-t4
!python -m ince_ayar siniflandirma tahmin --model ciktilar/siniflandirma-t4 --onnx --metin "Kargom hâlâ gelmedi"

## 4. Gömme modelini alana uydurma

Eğitimden önce ve sonra recall@k / MRR ölçülür; fark rapora yazılır.

In [ ]:
!python -m ince_ayar gomme egit --hazir colab-t4
!python -m ince_ayar gomme disa-aktar --hazir colab-t4

## 5. Sonuçları indir

Ara kayıtlar (`denetim/`) büyük olduğu için zip dışında bırakılır.

In [ ]:
!zip -qr ciktilar.zip ciktilar -x "*/denetim/*"
from google.colab import files
files.download("ciktilar.zip")